In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
import os
from matplotlib.colors import LinearSegmentedColormap
import warnings

from AnalysePoisson import *  
from AnalyseNegBinomial import *  
from scipy.optimize import curve_fit

# Load data

In [ ]:
Genta = pd.read_csv(f'Tables/Gentamicin_measured.csv',index_col=0)
Genta_threshold=[20,20,40,40]

Tetra = pd.read_csv(f'Tables/Tetracycline_measured.csv',index_col=0)
Tetra_threshold=[20,20,20,20]


Chp = pd.read_csv(f'Tables/Chloramphenicol_measured.csv',index_col=0)
Chp_threshold=[40,20,30,20,20,20,30,20,30,30,40,20,40,20]

Cipro = pd.read_csv(f'Tables/Ciprofloxacin_measured.csv',index_col=0)
Cipro_threshold=[20,20,20,20]


Genta.drop(Genta[Genta['trash']].index, inplace = True)
#Tetra.drop(Tetra[Tetra['trash']].index, inplace = True)
Chp.drop(Chp[Chp['trash']].index, inplace = True)
Cipro.drop(Cipro[Cipro['trash']].index, inplace = True)

Genta=Genta.reset_index()
#Tetra=Tetra.reset_index()
Chp=Chp.reset_index()
Cipro=Cipro.reset_index()

In [ ]:
Genta.drop(Genta[(Genta['concentration']==0.15) & (Genta['dataset']==1)].index, inplace = True)
Genta=Genta.reset_index()


In [ ]:
# here only the first  row are the fit parameters the rest is covariance matrix
Genta_f=pd.read_csv('Tables/Gentamicin_Fit.csv',index_col=0)
Chp_f=pd.read_csv('Tables/Chloramphenicol_Fit.csv',index_col=0)
Tetra_f=pd.read_csv('Tables/Tetracycline_Fit.csv',index_col=0)
Cipro_f=pd.read_csv('Tables/Ciprofloxacin_Fit.csv',index_col=0)

In [ ]:
xmax=[2.04,8.2,104.04,16.5]
xmin=[-0.08,-0.2,-4,-0.7]


# Run Analysis for a collection of thresholds

In [ ]:
def AnalyseSusceptibility(data,threshold,cond):

    
    data=data[data['n_cells']<threshold]
    data['dead']=data.n_cells_final < threshold
    data['dead'] =  data.dead.replace({True: 1, False: 0})
    
    metadata_cols = ['date']

    metadata = (
        data
        .groupby(['dataset', 'concentration'])[metadata_cols]
        .first()
        .reset_index()
    )

    chip_info = (
        data
        .groupby(['dataset', 'concentration'])
        .apply(AnalyseNegBinomChip)
        .droplevel(level=2)
        .reset_index()
    )
    # mergee metadata back
    chip_info = chip_info.merge(
        metadata,
        on=['dataset', 'concentration'],
        how='left'
    )
    
    
    
    
    qzero_threshold=0.6
    chip_info.drop(chip_info[chip_info['prob_neg_drop_zero_det']<qzero_threshold].index,inplace=True)
    

    if(cond=='Tetra'):
        y=chip_info.loc[chip_info.date==20230404,'q_chip_like'].values
        c=chip_info.loc[chip_info.date==20230404,'concentration'].values
        popt, pcov = curve_fit(lambda t, qz,a,b: 1- (1-qz)* np.exp(-np.power(t/a,b)), c,y,bounds=([0,0,0],[1,np.inf,np.inf]),p0=[0.3,66,3])

        y2=chip_info.loc[chip_info.date==20230315,'q_chip_like'].values
        c2=chip_info.loc[chip_info.date==20230315,'concentration'].values

        popt2, pcov2 = curve_fit(lambda t, qz,a,b: 1- (1-qz)* np.exp(-np.power(t/a,b)), c2,y2,bounds=([0,0,0],[1,np.inf,np.inf]),p0=[0.3,66,3])
        
        dictio= [{'q_0': popt[0],'a': popt[1],'b': popt[2],'threshold':threshold,'AB':cond+'-1'},
                {'q_0': popt2[0],'a': popt2[1],'b': popt2[2],'threshold':threshold,'AB':cond+'-2'}]

        
    else:    
        
        y=chip_info['q_chip_like'].values
        c=chip_info['concentration'].values
        popt, pcov = curve_fit(lambda t, qz,a,b: 1- (1-qz)* np.exp(-np.power(t/a,b)), c,y,bounds=([0,0,0],[1,np.inf,np.inf]),p0=[0.3,66,3])
        dictio= [{'q_0': popt[0],'a': popt[1],'b': popt[2],'threshold':threshold,'AB':cond}]
        
    return  dictio

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('ignore')

    thesholds_pos = [20,30,40,50]


    collect=[]
    for t in thesholds_pos:
        print(t)
        collect.extend(AnalyseSusceptibility(Genta,t,'Genta'))
        collect.extend(AnalyseSusceptibility(Tetra,t,'Tetra'))
        collect.extend(AnalyseSusceptibility(Chp,t,'Chp'))
        collect.extend(AnalyseSusceptibility(Cipro,t,'Cipro'))

    data=pd.DataFrame(collect)

# Plot 

In [ ]:
# Define once, use everywhere
standard_linewidth = 1.5  # Choose a value that looks good

spinesParams = {
    'axes.spines.right': True,
    'axes.spines.top': True,
    'axes.linewidth': standard_linewidth,
}



tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "serif",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 10,
    "font.size": 10,
    # Make the legend/label fonts a little smaller
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
}
tickParams = {
    "xtick.top":True,
    "xtick.bottom":True,
    "xtick.direction": "in",
    "ytick.left":True,
    "ytick.right":True,
    "ytick.direction": "in",
}
spinesParams = {
    'axes.spines.right' : True,
    'axes.spines.top' : True
}
plt.rcParams.update(tex_fonts)
plt.rcParams.update(tickParams)
plt.rcParams['text.latex.preamble'] = r'\usepackage{amsmath}\usepackage{amssymb}'
plt.rcParams.update(spinesParams)  

inchPerCm = 0.393701
goldenRatio = (1+np.sqrt(5))/2
figWidthCm = 17.9
#figHeightCm = figWidthCm
figHeightCm = figWidthCm*1.2

figWidthInches = figWidthCm*inchPerCm
figHeightInches = figHeightCm*inchPerCm


lw=1.5
marker_size=10


cmap = mpl.colormaps['cool']
cmap = LinearSegmentedColormap.from_list(
    'YlOrBr_dark',
    cmap(np.linspace(0.3, 1.0, 256))
)

threshold_colors = cmap(np.linspace(0, 1, 6))
colors=sns.color_palette("colorblind")


In [ ]:
def plot_threshold_dependence(
    data, data_main_fit, x_fitted, index, ax,
    cmap, main_color, line_width, ls='-',
    show_legend=False
):
    for i, t in enumerate(data.threshold.unique()):
        fit = data[data.threshold == t]

        label = f'{t}' if show_legend else None

        ax.plot(
            x_fitted,
            1-(1-fit['q_0'].values[0]) *
            np.exp(-np.power(x_fitted/fit['a'].values[0],
                             fit['b'].values[0])),
            color=cmap[i],
            linewidth=line_width,
            linestyle=ls,
            label=label
        )

    ax.plot(
        x_fitted,
        1-(1-data_main_fit['q_0'].values[index]) *
        np.exp(-np.power(x_fitted/data_main_fit['a'].values[index],
                         data_main_fit['b'].values[index])),
        color=main_color,
        linewidth=line_width,
        linestyle=ls
    )

In [ ]:
x_fitted = np.linspace(0, 150, 10000)
marker_size=5

fig_label_x_1=-0.17
fig_label_y_1=1

fig, ax = plt.subplots(2,2, figsize=(figWidthInches, figHeightInches))


plot_threshold_dependence(data[data.AB=='Genta'],Genta_f,x_fitted,0,ax[0,0],threshold_colors,(0.00392156862745098, 0.45098039215686275, 0.6980392156862745, 1.0),lw)
plot_threshold_dependence(data[data.AB=='Cipro'],Cipro_f,x_fitted,0,ax[0,1],threshold_colors,colors[1],lw,show_legend=True)
plot_threshold_dependence(data[data.AB=='Tetra-1'],Tetra_f,x_fitted,0,ax[1,0],threshold_colors,colors[3],lw)
plot_threshold_dependence(data[data.AB=='Tetra-2'],Tetra_f,x_fitted,4,ax[1,0],threshold_colors,colors[3],lw,'--')


plot_threshold_dependence(data[data.AB=='Chp'],Chp_f,x_fitted,0,ax[1,1],threshold_colors,colors[2],lw)




ax[0,0].set_ylabel('')
ax[0,0].set_xlabel(r'Gentamicin $\left[\frac{\mu g}{ml}\right]$',color=(0.00392156862745098, 0.45098039215686275, 0.6980392156862745, 1.0))


ax[0,1].set_ylabel('')
ax[0,1].set_xlabel(r'Ciprofloxacin $\left[\frac{n g}{ml}\right]$',color=colors[1])

ax[1,0].set_ylabel(r'q (single-cell susceptibility)',rotation=90)
ax[1,0].yaxis.set_label_coords(-0.15, 1.0)  # Position at middle of ax[0,0] and ax[1,0]

ax[1,0].set_xlabel(r'Tetracycline $\left[\frac{\mu g}{ml}\right]$',color=colors[3])



ax[1,1].set_ylabel('')
ax[1,1].set_xlabel(r'Chloramphenicol $\left[\frac{\mu g}{ml}\right]$',color=colors[2])



ax[0,0].set_xlim([xmin[0],xmax[0]])
ax[0,0].set_ylim([0,1.04])

ax[0,1].set_xlim([xmin[1],xmax[1]])
ax[0,1].set_ylim([0,1.04])

ax[1,0].set_xlim([xmin[2],xmax[2]])
ax[1,0].set_ylim([0,1.04])

ax[1,1].set_xlim([xmin[3],xmax[3]])
ax[1,1].set_ylim([0,1.04])

ax[0,1].legend(
        title='thresholds',
        loc="center left",
        bbox_to_anchor=(1.02, 0.8),
        frameon=True,
        borderaxespad=0
    )


# Adjust spacing - hspace controls vertical spacing between rows
plt.subplots_adjust(hspace=0.2)

plt.show()

In [ ]:
baseSavePath=''
saveFigPath = os.path.join(baseSavePath,'Figure_S4_sensitivity.pdf')
fig.savefig(saveFigPath,bbox_inches='tight')